# Part 3 — Greek Sensitivity Across Market Regimes

**Goal**: Quantify how the sensitivity of option Greeks (delta, gamma, vega, theta) to market moves changes across the four HMM-identified regimes.

**Model**: ΔGreek = α + β₁·ΔMarket + β₂·ΔVIX + Σγₖ·(Regimeₖ × ΔMarket) + ε

The γₖ coefficients measure additional Greek sensitivity vs. the calm baseline.

**Samples**:
- Primary: 20–45 DTE (active hedging window)
- Robustness: 5–20 DTE (FOMC window)

**Stratification**: Puts and calls analyzed separately. Moneyness buckets: deep OTM, OTM, ATM.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from panel import build_panel
from sensitivity import run_all, interaction_table
from plots import plot_sensitivity_heatmap, plot_put_call_comparison

DATA = '../data'

In [ ]:
# Load regime labels from Part 2
regime_labels = pd.read_parquet(f'{DATA}/regime_labels.parquet')
regime_labels['date'] = pd.to_datetime(regime_labels['date'])
print(regime_labels['regime'].value_counts())

# Load market features for regression covariates
ff3 = pd.read_parquet(f'{DATA}/ff3.parquet')
ff3['date'] = pd.to_datetime(ff3['date'])
vix = pd.read_parquet(f'{DATA}/cboeall1986.parquet')
vix['date'] = pd.to_datetime(vix['Date'])
market_features = ff3[['date','mktrf']].merge(vix[['date','vix']].dropna(), on='date', how='inner')
market_features = market_features[
    (market_features['date'] >= '1996-01-01') & (market_features['date'] <= '2019-12-31')
]

In [ ]:
# Build primary sample panel (20–45 DTE)
puts, calls = build_panel(
    f'{DATA}/spx_raw.parquet',
    regime_labels,
    market_features,
    dte_min=20,
    dte_max=45,
)
print(f'Puts panel: {len(puts):,} observations | {puts["contract_id"].nunique():,} contracts')
print(f'Calls panel: {len(calls):,} observations | {calls["contract_id"].nunique():,} contracts')

In [ ]:
# Run all regressions — puts
puts_results = run_all(puts)
puts_results.head(20)

In [ ]:
# Run all regressions — calls
calls_results = run_all(calls)
calls_results.head(20)

In [ ]:
# Interaction coefficient table (γ matrix)
print('PUTS — Interaction (γ) coefficients:')
display(interaction_table(puts_results))
print('\nCALLS — Interaction (γ) coefficients:')
display(interaction_table(calls_results))

In [ ]:
# Heatmaps — puts
for greek in ['delta', 'gamma', 'vega', 'theta']:
    fig = plot_sensitivity_heatmap(puts_results, greek, instrument='puts')
    plt.show()

In [ ]:
# Put vs. call comparison for gamma (the most hedging-relevant Greek)
for bucket in ['deep_otm', 'otm', 'atm']:
    fig = plot_put_call_comparison(puts_results, calls_results, 'gamma', bucket)
    plt.show()

In [ ]:
# Robustness: FOMC window (5–20 DTE)
puts_fomc, calls_fomc = build_panel(
    f'{DATA}/spx_raw.parquet',
    regime_labels,
    market_features,
    dte_min=5,
    dte_max=20,
)
print(f'FOMC puts: {len(puts_fomc):,} | calls: {len(calls_fomc):,}')
puts_fomc_results = run_all(puts_fomc)
calls_fomc_results = run_all(calls_fomc)
print('\nFOMC Puts γ:')
display(interaction_table(puts_fomc_results))